In [1]:
import json, time, os, re
import requests
import pandas as pd
import shutil

from enum import Enum
from typing import Any, List, Dict, Optional, Union

pd.set_option('display.max_columns', None)

SCRYFALL_BASE_URL = "https://api.scryfall.com/"
CARDS_COLUMNS = [
    'name',
    'oracle_id',
    # 'image_uris',
    'image_uri',
    'mana_cost',
    'cmc',
    'type_line',
    'oracle_text',
    'toughness',
    'colors',
    'color_identity',
    'keywords',
    # 'legalities',
    # 'set',
    # 'set_name',
    # 'scryfall_set_uri',
    # 'rulings_uri',
    # 'prints_search_uri',
    # 'flavor_text',
    'edhrec_rank',
    # 'prices',
    'price_usd',
]
os.getcwd()

'/Users/ptallo/Documents/mtg_data_science'

# Utils

In [2]:
def write_file(fpath: str, content: Union[str, bytes]) -> None:
    if not os.path.exists(os.path.dirname(fpath)):
        os.makedirs(os.path.dirname(fpath))

    if isinstance(content, str):
        with open(fpath, "wb") as f:
            f.write(content.encode("utf-8"))
    elif isinstance(content, bytes):
        with open(fpath, "wb") as f:
            f.write(content)
    
def read_file(fpath: str) -> Union[str, bytes]:
    with open(fpath, "rb") as f:
        b = f.read()
    return b.decode('utf-8') if fpath.endswith(('.json', '.csv', '.txt')) else b

# Caching and Requesting

In [3]:
class CacheType(Enum):
    JSON = "json"
    PNG = "png"

class CacheHandler:
    def __init__(self, cache_dir: str, cache_type: CacheType = CacheType.JSON):
        self.cache_dir = cache_dir
        self.cache_type = cache_type

    def _get_key_fpath(self, key: str) -> str:
        return os.path.join(self.cache_dir, f'{key}.{self.cache_type.value}')

    def get(self, key: str) -> Union[str, bytes, None]:
        fpath = self._get_key_fpath(key)
        return read_file(fpath) if os.path.exists(fpath) else None

    def set(self, key: str, value: Union[str, bytes]) -> None:
        fpath = self._get_key_fpath(key)
        write_file(fpath, value)

    def delete(self, key: str) -> None:
        fpath = self._get_key_fpath(key)
        print(fpath)
        if not os.path.exists(fpath):
            return
        os.remove(fpath)

    def clear(self) -> None:
        files = [x.split(".")[0] for x in os.listdir(self.cache_dir)]
        for f in files:
            self.delete(f)

In [4]:
class ScryfallHttpClient:
    def _get_scryfall_headers(self) -> Dict[str, str]:
        return {
            "User-Agent": "MTG-DataScience-Project/1.0",
            "Accept": "application/json;q=0.9,*/*,q=0.8",
        }
    
    def get(self, url: str, **kwargs) -> requests.Response:
        response = requests.get(url, headers=self._get_scryfall_headers(), **kwargs)
        if response.status_code != 200:
            response.raise_for_status()
        return response

In [ ]:
class BulkData(Enum):
    ORACLE_CARDS = "oracle_cards"
    UNIQUE_ARTWORK = "unqiue_artwork"
    DEFAULT_CARDS = "default_cards"
    ALL_CARDS = "all_cards"
    RULINGS = "rulings"
    ART_TAGS = "art_tags"
    ORACLE_TAGS = "oracle_tags"

class ScryfallDataHandler:
    def __init__(self, base_url: str, cache_handler: CacheHandler, http_client: ScryfallHttpClient):
        self.cache_handler = cache_handler
        self.http_client = http_client
        self.base_url = base_url
        self.endpoints = { "bulk-data": "bulk-data" }

    def _wrap_cache_contents(self, contents: str, json_key: str = None) -> pd.DataFrame:
        df = pd.DataFrame(json.loads(contents) if json_key is None else json.loads(contents).get(json_key))
        if 'image_uris' in df.columns:
            df['image_uri'] = df['image_uris'].apply(lambda x: x.get('normal') if isinstance(x, dict) else None)
        if 'prices' in df.columns:
            df['price_usd'] = df['prices'].apply(lambda x: x.get('usd') if isinstance(x, dict) else None)
        if 'legalities' in df.columns:
            df['commander_legality'] = df['legalities'].apply(lambda x: x.get('commander') if isinstance(x, dict) else None)
        return df

    def _get_bulk_data_download_link(self, bd: BulkData) -> str:
        df = self.get_bulk_data().copy()
        return df[df['type'].eq(bd.value)].download_uri.iloc[0]

    def get_bulk_data(self) -> pd.DataFrame:
        if self.cache_handler.get(self.endpoints['bulk-data']) is not None:
            return self._wrap_cache_contents(self.cache_handler.get(self.endpoints['bulk-data']), json_key="data")

        response_json = self.http_client.get(f"{self.base_url}/{self.endpoints['bulk-data']}")
        self.cache_handler.set(self.endpoints['bulk-data'], json.dumps(response_json.json(), indent=4))
        cache_contents = self.cache_handler.get(self.endpoints['bulk-data'])
        return pd.DataFrame(json.loads(cache_contents).get("data"))

    def get_cards_from_bulk_data(self, bd: BulkData) -> pd.DataFrame:
        if self.cache_handler.get(bd.value) is not None:
            return self._wrap_cache_contents(self.cache_handler.get(bd.value))

        response = self.http_client.get(self._get_bulk_data_download_link(bd), stream=True)
        self.cache_handler.set(bd.value, response.content)
        return self._wrap_cache_contents(self.cache_handler.get(bd.value))

# Fetching Images

In [6]:
json_cache_handler = CacheHandler("./cache/json/", cache_type=CacheType.JSON)
png_cache_handler = CacheHandler("./cache/png/", cache_type=CacheType.PNG)
scryfall_http_client = ScryfallHttpClient()
sfdh = ScryfallDataHandler(SCRYFALL_BASE_URL, json_cache_handler, scryfall_http_client)

# png_cache_handler.clear()
# json_cache_handler.clear()

In [7]:
cards_df = sfdh.get_cards_from_bulk_data(BulkData.ORACLE_CARDS)
cards_df.shape

AttributeError: 'list' object has no attribute 'get'

In [ ]:
sorted([x for x in cards_df.columns if 'legal' in x])

In [ ]:
cards_df[[
    'oracle_id', 
    'name', 
    
    'cmc',
    'mana_cost',
    'color_identity',
    'colors',
    'type_line',
    'oracle_text',
    'power',
    'toughness',
    'keywords',
    'game_changer',
    'defense',
    'hand_modifier',
    'life_modifier',
    'loyalty',
    'produced_mana',
    'reserved',
    'commander_legality',

    'price_usd',
    'penny_rank',
    'edhrec_rank',
]].head()